# Q2: Unit Cost Forecasting
## Can we forecast the price per unit from supplier and order characteristics?

**Report Section:** Question 2 · Regression  
**Data Source:** Supply Chain ML Predictive Modelling Report

## Objective

- Build a regression model to predict `price_per_unit` — enabling proactive cost management before procurement
- Engineer pricing signals that encode business logic (e.g., fulfillment ratio, demand pressure)
- Simulate procurement savings by delivery term and action code to quantify model business value in dollar terms

In [ ]:
# Step 0: Setup & Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load dataset
# df = pd.read_csv('supply_chain_data.csv')  # Replace with actual data path
# print(df.head())
# print(df.info())
# print(df['price_per_unit'].describe())

## Implementation Plan

### Step 1: Explore Price Distribution & Detect Outliers
- Plot histogram and box plot of `price_per_unit`
- Identify outliers using IQR method (1.5× IQR rule)
- Document skewness and kurtosis; consider log-transform if right-skewed

### Step 2: Feature Engineering — Build Pricing Signals
- **fulfillment_ratio** = items_offered / items_requested (supplier coverage)
- **demand_pressure** = demand_volatility_index × seasonality_index (combined stress)
- Include: quality_score, supplier_reliability_score, payment_term_days, offer_validity_days, delivery_term_code, procurement_action_code
- Create correlation heatmap vs. price_per_unit

### Step 3: Train-Test Split & Gradient Boosting Regressor
- 80/20 split with random_state=42
- Build pipeline: ColumnTransformer (OneHotEncoder + StandardScaler) → GradientBoostingRegressor
- Evaluate MAE, RMSE, R² on test set
- Visualize Predicted vs. Actual with 45° reference line

### Step 4: Cross-Validation & Hyperparameter Sensitivity
- Run 5-fold cross_val_score on full pipeline
- Test hyperparameter grid: n_estimators ∈ {100, 200, 300}, max_depth ∈ {3, 4, 5}
- Plot heatmap of CV R² across combinations

### Step 5: SHAP-Style Feature Contribution Analysis
- Compute permutation_importance on test set
- Plot ranked horizontal bar chart of mean importance ± std
- Identify dominant pricing drivers (quality, delivery term, demand pressure)

### Step 6: Procurement Savings Simulation & Final Scorecard
- For each test row: savings = actual_price - predicted_price
- Aggregate by delivery_term_code and procurement_action_code
- Plot grouped bar chart of average savings per strategy
- Build final scorecard table with MAE, RMSE, R², CV R² Mean, CV R² Std

In [ ]:
# STEP 1: Explore Price Distribution & Detect Outliers

# price = df['price_per_unit']

# # Calculate outlier bounds
# Q1 = price.quantile(0.25)
# Q3 = price.quantile(0.75)
# IQR = Q3 - Q1
# lower_bound = Q1 - 1.5 * IQR
# upper_bound = Q3 + 1.5 * IQR

# outliers = price[(price < lower_bound) | (price > upper_bound)]
# print(f"Total rows: {len(price)}")
# print(f"Outliers detected: {len(outliers)} ({len(outliers)/len(price)*100:.1f}%)")
# print(f"\nPrice statistics:")
# print(f"  Mean: {price.mean():.2f}")
# print(f"  Median: {price.median():.2f}")
# print(f"  Std Dev: {price.std():.2f}")
# print(f"  Skewness: {price.skew():.3f}")
# print(f"  Kurtosis: {price.kurtosis():.3f}")

In [ ]:
# STEP 1 Visualization: Price Distribution

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Histogram with KDE
# axes[0].hist(price, bins=40, edgecolor='black', alpha=0.7, density=True)
# price.plot.kde(ax=axes[0], color='red', linewidth=2, label='KDE')
# axes[0].set_xlabel('Price per Unit')
# axes[0].set_ylabel('Density')
# axes[0].set_title('Price Distribution (Histogram + KDE)')
# axes[0].legend()

# # Box plot with outlier annotation
# bp = axes[1].boxplot(price, vert=True, patch_artist=True)
# bp['boxes'][0].set_facecolor('lightblue')
# axes[1].set_ylabel('Price per Unit')
# axes[1].set_title(f'Box Plot (Outliers: {len(outliers)}, IQR bounds shown)')
# axes[1].grid(axis='y', alpha=0.3)

# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 2: Feature Engineering — Build Pricing Signals

# # Create engineered features
# df['fulfillment_ratio'] = df['items_offered'] / (df['items_requested'] + 1)  # Avoid division by zero
# df['demand_pressure'] = df['demand_volatility_index'] * df['seasonality_index']

# # Select features for modeling
# numerical_features = [
#     'quality_score', 'supplier_reliability_score', 'payment_term_days',
#     'offer_validity_days', 'fulfillment_ratio', 'demand_pressure'
# ]
# categorical_features = ['delivery_term_code', 'procurement_action_code']

# all_features = numerical_features + categorical_features

# # Correlation with price
# correlation_data = df[numerical_features + ['price_per_unit']].corr()['price_per_unit'].drop('price_per_unit').sort_values()
# print("\nCorrelation with price_per_unit:")
# print(correlation_data)

# # Heatmap
# fig, ax = plt.subplots(figsize=(10, 6))
# corr_matrix = df[numerical_features + ['price_per_unit']].corr()
# sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
# ax.set_title('Feature Correlation with Price per Unit')
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 3: Train-Test Split & Gradient Boosting Regressor

# X = df[all_features]
# y = df['price_per_unit']

# # Train-test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Create preprocessing pipeline
# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', StandardScaler(), numerical_features),
#         ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
#     ])

# # Gradient Boosting Regressor Pipeline
# gbr_pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('model', GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42))
# ])

# gbr_pipeline.fit(X_train, y_train)
# y_pred_gbr = gbr_pipeline.predict(X_test)

# # Metrics
# mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
# rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
# r2_gbr = r2_score(y_test, y_pred_gbr)

# print(f"Gradient Boosting Regressor Results:")
# print(f"  MAE:  {mae_gbr:.3f}")
# print(f"  RMSE: {rmse_gbr:.3f}")
# print(f"  R²:   {r2_gbr:.3f}")

In [ ]:
# STEP 3 Visualization: Predicted vs. Actual (GBR)

# fig, ax = plt.subplots(1, 1, figsize=(10, 8))
# ax.scatter(y_test, y_pred_gbr, alpha=0.6, s=50)
# min_val = min(y_test.min(), y_pred_gbr.min())
# max_val = max(y_test.max(), y_pred_gbr.max())
# ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
# ax.set_xlabel('Actual Price per Unit')
# ax.set_ylabel('Predicted Price per Unit')
# ax.set_title(f'Gradient Boosting: Predicted vs. Actual (R² = {r2_gbr:.3f})')
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 4: Cross-Validation & Hyperparameter Sensitivity

# # 5-fold cross-validation
# cv_scores = cross_val_score(gbr_pipeline, X_train, y_train, cv=5, scoring='r2')
# print(f"\nCross-Validation R² (5-fold):")
# print(f"  Mean: {cv_scores.mean():.3f}")
# print(f"  Std:  {cv_scores.std():.3f}")
# print(f"  Min:  {cv_scores.min():.3f}")
# print(f"  Max:  {cv_scores.max():.3f}")

# # Hyperparameter grid search (simplified)
# cv_results = {}
# for n_est in [100, 200, 300]:
#     for depth in [3, 4, 5]:
#         gbr_test = Pipeline([
#             ('preprocessor', preprocessor),
#             ('model', GradientBoostingRegressor(n_estimators=n_est, learning_rate=0.05, max_depth=depth, random_state=42))
#         ])
#         cv_score = cross_val_score(gbr_test, X_train, y_train, cv=3, scoring='r2').mean()
#         cv_results[(n_est, depth)] = cv_score

# # Reshape for heatmap
# cv_matrix = pd.DataFrame(
#     [[cv_results.get((n, d), 0) for d in [3, 4, 5]] for n in [100, 200, 300]],
#     index=[100, 200, 300],
#     columns=[3, 4, 5]
# )
# cv_matrix.index.name = 'n_estimators'
# cv_matrix.columns.name = 'max_depth'

# print("\nHyperparameter CV R² Grid:")
# print(cv_matrix)

In [ ]:
# STEP 4 Visualization: Hyperparameter Heatmap

# fig, ax = plt.subplots(figsize=(10, 6))
# sns.heatmap(cv_matrix, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5, ax=ax, cbar_kws={'label': 'R² Score'})
# ax.set_title('Cross-Validation R² Across Hyperparameter Grid')
# ax.set_xlabel('max_depth')
# ax.set_ylabel('n_estimators')
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 5: SHAP-Style Feature Contribution Analysis (Permutation Importance)

# gbr_model = gbr_pipeline.named_steps['model']
# perm_importance = permutation_importance(
#     gbr_pipeline, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
# )

# # Get feature names after preprocessing
# # (This is simplified; adjust based on your actual feature names after encoding)
# feature_names = all_features

# importance_df = pd.DataFrame({
#     'Feature': feature_names,
#     'Importance': perm_importance.importances_mean,
#     'Std': perm_importance.importances_std
# }).sort_values('Importance', ascending=True).tail(10)

# fig, ax = plt.subplots(figsize=(10, 6))
# ax.barh(importance_df['Feature'], importance_df['Importance'], xerr=importance_df['Std'], color='steelblue', capsize=5)
# ax.set_xlabel('Permutation Importance ± Std')
# ax.set_title('Top 10 Features Driving Price per Unit')
# plt.tight_layout()
# plt.show()

# print(importance_df)

In [ ]:
# STEP 6: Procurement Savings Simulation & Final Scorecard

# # Create a results dataframe for analysis
# results_df = pd.DataFrame({
#     'actual_price': y_test.values,
#     'predicted_price': y_pred_gbr,
#     'delivery_term_code': X_test['delivery_term_code'].values,
#     'procurement_action_code': X_test['procurement_action_code'].values
# })

# # Calculate savings (positive = actual was cheaper than predicted)
# results_df['savings'] = results_df['actual_price'] - results_df['predicted_price']

# # Aggregate by delivery term
# savings_by_term = results_df.groupby('delivery_term_code')['savings'].mean().sort_values(ascending=False)
# savings_by_action = results_df.groupby('procurement_action_code')['savings'].mean().sort_values(ascending=False)

# print("\nAverage Savings by Delivery Term:")
# print(savings_by_term)
# print("\nAverage Savings by Procurement Action:")
# print(savings_by_action)

In [ ]:
# STEP 6 Visualization: Savings by Strategy

# fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# # Savings by delivery term
# savings_by_term.plot(kind='barh', ax=axes[0], color='steelblue')
# axes[0].set_xlabel('Average Savings per Unit')
# axes[0].set_title('Procurement Savings by Delivery Term')
# axes[0].grid(axis='x', alpha=0.3)

# # Savings by procurement action
# savings_by_action.plot(kind='barh', ax=axes[1], color='darkgreen')
# axes[1].set_xlabel('Average Savings per Unit')
# axes[1].set_title('Procurement Savings by Procurement Action')
# axes[1].grid(axis='x', alpha=0.3)

# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 6: Final Model Scorecard

# scorecard = pd.DataFrame({
#     'Metric': ['MAE', 'RMSE', 'R² (Test Set)', 'CV R² (Mean)', 'CV R² (Std)'],
#     'Value': [
#         f"{mae_gbr:.3f}",
#         f"{rmse_gbr:.3f}",
#         f"{r2_gbr:.3f}",
#         f"{cv_scores.mean():.3f}",
#         f"{cv_scores.std():.3f}"
#     ]
# })

# print("\n" + "="*50)
# print("GRADIENT BOOSTING REGRESSOR - FINAL SCORECARD")
# print("="*50)
# print(scorecard.to_string(index=False))

## Key Takeaways

Once all steps are complete, your notebook will deliver:

1. **Price Distribution Insight** — Outlier detection and skewness analysis
2. **Engineering Impact** — fulfillment_ratio and demand_pressure as strong pricing signals
3. **Model Performance** — GBR MAE, RMSE, R² with cross-validation stability check
4. **Pricing Drivers** — Top 5 features that suppliers actually price into their quotes (quality, delivery term, demand pressure)
5. **Financial Impact** — Dollar savings by delivery term and procurement action strategy

The savings simulation converts accuracy into procurement budget language — the only metric that drives policy reform.